In [ ]:
import cv2
import torch
import numpy as np
import mediapipe as mp

# -------------------------------
# 1. Load model
# -------------------------------
class FingerMLP(torch.nn.Module):
    def __init__(self, input_dim=126, hidden_dim=256, num_classes=6):
        super(FingerMLP, self).__init__()
        self.fc1 = torch.nn.Linear(input_dim, hidden_dim)
        self.bn1 = torch.nn.LayerNorm(hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = torch.nn.LayerNorm(hidden_dim)
        self.relu = torch.nn.ReLU()
        self.dropout = torch.nn.Dropout(0.3)
        self.fc_out = torch.nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(self.relu(self.bn2(self.fc2(x))))
        return self.fc_out(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FingerMLP().to(device)
model.load_state_dict(torch.load(r"C:\Users\garba\Downloads\processed\finger_mlp_model.pth", map_location=device))
model.eval()

# -------------------------------
# 2. Mediapipe setup
# -------------------------------
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5)

def normalize_landmarks(lm):
    lh = lm[:63].reshape(21,3)
    rh = lm[63:].reshape(21,3)
    if lh.sum() != 0:
        lh -= lh[0]
        lh /= np.linalg.norm(lh, axis=0).max()
    if rh.sum() != 0:
        rh -= rh[0]
        rh /= np.linalg.norm(rh, axis=0).max()
    return np.concatenate([lh.flatten(), rh.flatten()])

def extract_hand_landmarks(results):
    landmarks = np.zeros(126)
    if results.multi_hand_landmarks:
        for idx, hand_landmarks in enumerate(results.multi_hand_landmarks):
            hand_label = results.multi_handedness[idx].classification[0].label
            coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]).flatten()
            if hand_label == "Left":
                landmarks[:63] = coords
            else:
                landmarks[63:] = coords
    return normalize_landmarks(landmarks)

# -------------------------------
# 3. Real-time webcam
# -------------------------------
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Flip frame for selfie-view
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(rgb_frame)

    # Draw hand landmarks only
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        # Prepare input for model
        lm_array = extract_hand_landmarks(results)
        lm_tensor = torch.tensor(lm_array, dtype=torch.float32).unsqueeze(0).to(device)

        # Predict finger count
        with torch.no_grad():
            out = model(lm_tensor)
            pred = torch.argmax(out, dim=1).item()

        cv2.putText(frame, f"Fingers: {pred}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,255,0), 3)

    cv2.imshow("Hand Finger Count", frame)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC key to exit
        break

cap.release()
cv2.destroyAllWindows()
